# Outlier Inspector (이상치 분석 도구)

이 노트북은 특정 `game_id`와 `action_id`를 기준으로 해당 액션의 앞뒤 10개 액션(총 21개)을 보여주어, 데이터 이상치(예: 순간 이동)의 원인을 맥락적으로 파악하기 위해 작성되었습니다.

In [14]:
import pandas as pd
import numpy as np
import os

# 데이터 로드
RAW_DATA_PATH = '../data/refined/v5.3/preprocessed_data.csv'
df = pd.read_csv(RAW_DATA_PATH, encoding='utf-8-sig')
print(f"Data loaded: {len(df):,} rows")

Data loaded: 465,787 rows


## 1. 검색 조건 설정
분석하고 싶은 `target_game_id`와 `target_action_id`를 입력하세요.

In [15]:
target_game_id = 126313     # 예시
target_action_id =  36   # 예시
window_size = 10         # 앞뒤로 볼 액션 수

In [ ]:
# 해당 게임의 데이터 추출
game_df = df[df['game_id'] == target_game_id].sort_values(['period_id', 'time_seconds', 'action_id']).reset_index(drop=True)

# target_action_id의 현재 인덱스 찾기
try:
    idx = game_df[game_df['action_id'] == target_action_id].index[0]
    
    start_idx = max(0, idx - window_size)
    end_idx = min(len(game_df), idx + window_size + 1)
    
    context_df = game_df.iloc[start_idx:end_idx].copy()
    
    # 속도 관련 계산 (로컬)
    context_df['prev_end_x'] = context_df['end_x'].shift(1)
    context_df['prev_end_y'] = context_df['end_y'].shift(1)
    context_df['prev_time'] = context_df['time_seconds'].shift(1)
    
    # 이벤트 간 이동 거리 및 속도
    context_df['gap_dist'] = np.sqrt((context_df['start_x'] - context_df['prev_end_x'])**2 + (context_df['start_y'] - context_df['prev_end_y'])**2)
    context_df['gap_dt'] = context_df['time_seconds'] - context_df['prev_time']
    context_df['gap_speed'] = context_df['gap_dist'] / context_df['gap_dt'].replace(0, np.nan)

    # 결과 출력
    cols = [ 'time_seconds', 'player_name_ko', 'type_name', 'start_x', 'start_y', 'end_x', 'end_y', 'gap_speed']
    display(context_df[cols].style.apply(lambda x: ['background: orange; color: black' if v == target_action_id else '' for v in context_df['action_id']], axis=0))

except IndexError:
    print(f"Error: game_id {target_game_id}에서 action_id {target_action_id}를 찾을 수 없습니다.")

,time_seconds,player_name_ko,type_name,start_x,start_y,end_x,end_y,gap_speed
17,21.233000,설영우,Carry,0.440430,0.061706,0.432140,0.091821,nan
18,22.219000,설영우,Pass,0.432140,0.091821,0.480813,0.169463,0.000000
19,22.983000,보야니치,Pass,0.480813,0.169463,0.401078,0.124560,0.000000
20,24.168000,설영우,Carry,0.401078,0.124560,0.382844,0.180121,0.000000
21,26.816000,설영우,Pass,0.382844,0.180121,0.528877,0.010938,0.000000
22,28.334000,이청용,Carry,0.528877,0.010938,0.434258,0.239489,0.000000
23,31.998000,이청용,Pass,0.434258,0.239489,0.308130,0.365534,0.000000
24,33.168000,임종은,Carry,0.308130,0.365534,0.293376,0.431856,0.000000
25,34.504000,임종은,Pass,0.293376,0.431856,0.321657,0.738111,0.000000
26,36.061000,황석호,Carry,0.321657,0.738111,0.334952,0.789240,0.000000


: 